# EdgeSentiment - Colab Training (DistilBERT on SST-2)

This notebook fine-tunes `distilbert-base-uncased` for 2-class sentiment
classification on the SST-2 (Stanford Sentiment Treebank) subset of GLUE. It is
the **self-contained, GPU-ready** version of `train.py` and is meant to run on a
**Google Colab T4 GPU** (`Runtime > Change runtime type > T4 GPU`).

When it finishes you will have a trained model in your Google Drive plus a
`model.zip` you can download and unzip into `training/models/` of the project.

**Expected runtime:** ~15-20 minutes on T4. **Target validation accuracy:** >91%.

## 1. Install dependencies

Install the Hugging Face stack (`transformers`, `datasets`, `accelerate`,
`evaluate`) plus the ONNX tooling that downstream stages use. We install the
ONNX packages here too so the saved environment is consistent end-to-end.

In [ ]:
!pip install -q transformers==4.44.2 datasets==2.21.0 torch onnx==1.16.2 onnxruntime==1.19.2 onnxruntime-tools evaluate==0.4.3 accelerate==0.34.2 scikit-learn==1.5.1

## 2. Mount Google Drive

We save the trained model into Google Drive so it survives the Colab session
and can be downloaded afterwards. Mounting will prompt you to authorize access.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Configuration and imports

Hyperparameters match `train.py` exactly (3 epochs, batch size 16, lr 2e-5,
weight decay 0.01). The output directory points at Drive so the model persists.
We confirm a CUDA GPU is visible - if this prints `False`, switch the runtime to
T4 GPU before continuing.

In [ ]:
from pathlib import Path
import json
import numpy as np
import torch
import evaluate
from datasets import load_dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

MODEL_NAME = 'distilbert-base-uncased'
MAX_SEQ_LENGTH = 128
TARGET_ACCURACY = 0.91

OUTPUT_DIR = Path('/content/drive/MyDrive/edge-sentiment/models/distilbert-sst2-finetuned')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

NUM_EPOCHS = 3
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
SEED = 42

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 4. Load and tokenize SST-2

Load the GLUE SST-2 train/validation splits and tokenize the `sentence` field.
We truncate to 128 tokens and leave padding to a dynamic collator so each batch
pads only to its longest example (faster than padding everything to 128).

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch['sentence'], truncation=True, max_length=MAX_SEQ_LENGTH)

raw = load_dataset('glue', 'sst2')
train_ds = raw['train'].map(tokenize, batched=True)
eval_ds = raw['validation'].map(tokenize, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print('Train examples:', len(train_ds), '| Validation examples:', len(eval_ds))

## 5. Build the model

Load `distilbert-base-uncased` with a fresh 2-label classification head and
human-readable `id2label` / `label2id` mappings (negative=0, positive=1) so the
exported model is self-describing.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: 'negative', 1: 'positive'},
    label2id={'negative': 0, 'positive': 1},
)

## 6. Fine-tune with the Trainer API

Standard Hugging Face `Trainer` loop: evaluate every epoch, keep the best
checkpoint by accuracy, enable mixed-precision (`fp16`) on the GPU for speed.
Accuracy is computed with the `evaluate` library.

In [ ]:
accuracy_metric = evaluate.load('accuracy')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple):
        logits = logits[0]
    preds = np.argmax(logits, axis=-1)
    return {'accuracy': float(accuracy_metric.compute(predictions=preds, references=labels)['accuracy'])}

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / 'checkpoints'),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=64,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_strategy='steps',
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    greater_is_better=True,
    seed=SEED,
    report_to=[],
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

## 7. Evaluate, save model, and report accuracy

Run a final evaluation, save the best model + tokenizer to Drive, and dump the
training log history (for the loss-curve plot in `walkthrough.ipynb`). The final
validation accuracy is printed prominently and checked against the >91% target.

In [ ]:
metrics = trainer.evaluate()
accuracy = float(metrics['eval_accuracy'])

trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))
(OUTPUT_DIR / 'training_log.json').write_text(json.dumps({
    'final_accuracy': accuracy,
    'log_history': trainer.state.log_history,
}, indent=2))

print('\n' + '=' * 60)
print(f'FINAL VALIDATION ACCURACY: {accuracy * 100:.2f}%')
print(f'TARGET:                    {TARGET_ACCURACY * 100:.2f}%')
print('PASSED' if accuracy >= TARGET_ACCURACY else 'BELOW TARGET - investigate')
print('=' * 60)

## 8. Zip the model for download

Package the model directory into `model.zip` in your Drive. After this cell
finishes, download `MyDrive/edge-sentiment/model.zip`, unzip it, and place the
`distilbert-sst2-finetuned/` folder into `training/models/` of the project.

In [ ]:
!zip -r /content/drive/MyDrive/edge-sentiment/model.zip /content/drive/MyDrive/edge-sentiment/models/
print('\nDone. Download MyDrive/edge-sentiment/model.zip from Google Drive.')